# GANForge -- GAN (Standalone Notebook)

**Unconditional generation from noise**

This is a **self-contained extract** of the GAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="1"></a>
## 1. Original GAN -- Unconditional generation from noise

### 1.1 Problem Statement & Why this GAN

Goodfellow et al. (2014) posed generative modeling as a **two-player
minimax game**: a *Generator* `G` maps random noise `z ~ N(0, I)` to samples
that try to fool a *Discriminator* `D`, while `D` tries to tell real data
from `G`'s fakes. Neither network is ever told *what* to draw -- there is no
label, no condition, nothing but noise in and an image out. This is the
**unconditional generation** problem: learn the data distribution `p(x)`
well enough to sample convincingly from it.

We implement a DCGAN-style (Radford et al., 2015) convolutional
Generator/Discriminator pair -- the architecture that made the original
minimax idea actually stable enough to train reliably -- on MNIST digits.

💡 **Insight.** Every other GAN in this notebook is a variation on this
same adversarial game: cGAN/AC-GAN add conditioning, ProGAN/StyleGAN scale
the architecture, CycleGAN/SRGAN change what "real vs fake" is computed
over. Understanding this section well makes every later section a diff,
not a rewrite.

### 1.2 Data Collection & Preparation

In [ ]:
# MNIST is tiny (~11MB), auto-downloads via torchvision, and trains fast --
# ideal for demonstrating the *original* unconditional GAN game clearly and
# quickly. We normalize to [-1, 1] because our Generator's final activation
# is Tanh (standard DCGAN choice: bounded, zero-centered output).
GAN_IMG_SIZE = 32       # upsampled from MNIST's native 28x28 for clean conv strides (32 = 2^5)
GAN_BATCH_SIZE = 128
GAN_NZ = 100             # latent noise dimensionality
GAN_EPOCHS = 5            # small on purpose -- see Section 0 compute warning; raise for better samples

gan_transform = T.Compose([
    T.Resize(GAN_IMG_SIZE),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

try:
    gan_dataset = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=gan_transform)
    print(f"Loaded real MNIST: {len(gan_dataset)} images")
except Exception as e:
    # Offline fallback: procedurally generated blob "digits" so the rest of
    # the section still runs end-to-end without internet access.
    print(f"MNIST download failed ({e}); falling back to synthetic blobs.")
    rng = np.random.default_rng(SEED)
    fake_imgs = torch.zeros(2000, 1, GAN_IMG_SIZE, GAN_IMG_SIZE)
    yy, xx = torch.meshgrid(torch.arange(GAN_IMG_SIZE), torch.arange(GAN_IMG_SIZE), indexing="ij")
    for i in range(2000):
        cx, cy = rng.uniform(8, 24, 2)
        r = rng.uniform(4, 10)
        blob = ((xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2).float()
        fake_imgs[i, 0] = blob * 2 - 1
    gan_dataset = TensorDataset(fake_imgs, torch.zeros(2000, dtype=torch.long))

gan_loader = DataLoader(gan_dataset, batch_size=GAN_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
real_batch, _ = next(iter(gan_loader))
show_images(real_batch[:64], title="Real MNIST samples", nrow=8)


### 1.3 Model Architecture

In [ ]:
class GAN_Generator(nn.Module):
    '''DCGAN generator: a stack of transposed convolutions that upsample a
    1x1 noise vector into a full image, doubling spatial size each layer.
    BatchNorm + ReLU everywhere except the output (Tanh, to match our
    [-1, 1] data normalization) is the classic DCGAN recipe for stable
    gradients through a deep generator.'''
    def __init__(self, nz=GAN_NZ, ngf=64, nc=1):
        super().__init__()
        self.net = nn.Sequential(
            # z: (nz, 1, 1) -> (ngf*4, 4, 4)
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            # -> (ngf*2, 8, 8)
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            # -> (ngf, 16, 16)
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            # -> (nc, 32, 32)
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class GAN_Discriminator(nn.Module):
    '''Mirror-image convolutional binary classifier: real (1) vs fake (0).
    LeakyReLU (not ReLU) is used throughout -- DCGAN found this crucial for
    keeping gradients flowing to the generator through the discriminator's
    negative activations, avoiding dead-neuron style discriminator collapse.'''
    def __init__(self, nc=1, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(-1)  # raw logits (no Sigmoid -- we use BCEWithLogitsLoss for numerical stability)


nc_gan = real_batch.shape[1]
G_gan = GAN_Generator(nc=nc_gan).to(DEVICE).apply(weights_init)
D_gan = GAN_Discriminator(nc=nc_gan).to(DEVICE).apply(weights_init)
print(G_gan)
print(D_gan)


### 1.4 Loss Functions & Optimizers

In [ ]:
# Non-saturating minimax loss (Goodfellow et al.): the generator maximizes
# log(D(G(z))) instead of minimizing log(1 - D(G(z))) -- same fixed point,
# but far stronger gradients early in training when D easily rejects G.
gan_criterion = nn.BCEWithLogitsLoss()

# Adam with beta1=0.5 (not the default 0.9) is the standard DCGAN choice:
# high beta1 momentum tends to cause oscillation/instability in the
# adversarial min-max setting.
gan_opt_G = torch.optim.Adam(G_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))
gan_opt_D = torch.optim.Adam(D_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))

GAN_REAL_LABEL, GAN_FAKE_LABEL = 1.0, 0.0
gan_fixed_noise = torch.randn(64, GAN_NZ, device=DEVICE)  # fixed across training to visualize learning progress on the SAME latents


### 1.5 Training Loop

In [ ]:
gan_scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
gan_scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)
gan_loss_hist = {"G": AvgMeter(), "D": AvgMeter()}
gan_history = []

for epoch in range(GAN_EPOCHS):
    pbar = tqdm(gan_loader, desc=f"[GAN] epoch {epoch+1}/{GAN_EPOCHS}")
    for real, _ in pbar:
        real = real.to(DEVICE)
        b = real.size(0)

        # ---- Train Discriminator: maximize log(D(real)) + log(1 - D(G(z))) ----
        D_gan.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits_real = D_gan(real)
            loss_d_real = gan_criterion(logits_real, torch.full((b,), GAN_REAL_LABEL, device=DEVICE))
            noise = torch.randn(b, GAN_NZ, device=DEVICE)
            fake = G_gan(noise)
            logits_fake = D_gan(fake.detach())
            loss_d_fake = gan_criterion(logits_fake, torch.full((b,), GAN_FAKE_LABEL, device=DEVICE))
            loss_d = loss_d_real + loss_d_fake
        gan_scaler_D.scale(loss_d).backward()
        gan_scaler_D.step(gan_opt_D)
        gan_scaler_D.update()

        # ---- Train Generator: maximize log(D(G(z))) (non-saturating trick) ----
        G_gan.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits_fake_for_g = D_gan(fake)
            loss_g = gan_criterion(logits_fake_for_g, torch.full((b,), GAN_REAL_LABEL, device=DEVICE))
        gan_scaler_G.scale(loss_g).backward()
        gan_scaler_G.step(gan_opt_G)
        gan_scaler_G.update()

        gan_loss_hist["D"].update(loss_d.item())
        gan_loss_hist["G"].update(loss_g.item())
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{loss_g.item():.3f}")

    gan_history.append((gan_loss_hist["G"].avg, gan_loss_hist["D"].avg))
    with torch.no_grad():
        G_gan.eval()
        sample = G_gan(gan_fixed_noise)
        G_gan.train()
    show_images(sample, title=f"GAN samples after epoch {epoch+1}")
    save_checkpoint({"G": G_gan.state_dict(), "D": D_gan.state_dict(), "epoch": epoch},
                     f"{CHECKPOINT_DIR}/gan_epoch{epoch+1}.pt")

print("Original GAN training complete.")


### 1.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def gan_generate(n_samples: int = 16, seed: int = None) -> torch.Tensor:
    '''Clean generation entry point: noise-in, images-out ([-1, 1] range).'''
    if seed is not None:
        torch.manual_seed(seed)
    G_gan.eval()
    z = torch.randn(n_samples, GAN_NZ, device=DEVICE)
    imgs = G_gan(z)
    G_gan.train()
    return imgs

show_images(gan_generate(16), title="GAN inference: fresh samples", nrow=4)


### 1.7 Visualization & Results

In [ ]:
# Loss curves: in a healthy GAN neither loss collapses to 0 -- both hover
# because D and G are constantly adapting to each other.
gs, ds = zip(*gan_history)
plt.figure(figsize=(6, 4))
plt.plot(gs, label="Generator loss")
plt.plot(ds, label="Discriminator loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("GAN training losses")
plt.show()

# Latent-space interpolation: walking a straight line in z-space and
# decoding each point shows whether the generator learned a *smooth*
# manifold (no sudden jumps) rather than memorizing isolated points.
@torch.no_grad()
def gan_interpolate(n_steps: int = 8):
    G_gan.eval()
    z1, z2 = torch.randn(1, GAN_NZ, device=DEVICE), torch.randn(1, GAN_NZ, device=DEVICE)
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE).view(-1, 1)
    z_interp = (1 - alphas) * z1 + alphas * z2
    imgs = G_gan(z_interp)
    G_gan.train()
    return imgs

show_images(gan_interpolate(8), title="Latent interpolation", nrow=8, figsize=(12, 2))


### 1.8 User Input → Output Demo

In [ ]:
# "Interactive" cell: change N_SAMPLES_TO_GENERATE and RANDOM_SEED, re-run.
N_SAMPLES_TO_GENERATE = 25
RANDOM_SEED = 7

demo_samples = gan_generate(N_SAMPLES_TO_GENERATE, seed=RANDOM_SEED)
show_images(demo_samples, title=f"User demo: {N_SAMPLES_TO_GENERATE} samples, seed={RANDOM_SEED}", nrow=5)


### 1.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** The non-saturating generator loss is what makes vanilla
  GAN training practical at all -- the original minimax loss gives
  vanishing gradients to `G` whenever `D` is confident, which is almost
  always early in training.
- ⚠️ **Warning.** No conditioning means you cannot ask this model for "a
  7" -- you only get a random sample from the learned distribution. This
  exact limitation motivates cGAN (next section).
- ⚠️ **Warning.** Vanilla DCGAN is prone to **mode collapse** (generator
  finds a few outputs that reliably fool D and stops exploring) and has no
  formal convergence guarantee -- both are addressed by later
  architectures in this notebook (WGAN-GP-style losses appear in
  StackGAN/BigGAN sections).

**This GAN solved the problem of: unconditional image generation --
learning to sample realistically from an unlabeled image distribution
given nothing but random noise.**